In [1]:
# Single-cell Google Colab example
%pip install -q -U google-genai

from getpass import getpass
from google import genai

#client = genai.Client(
#    api_key=getpass("Enter your Gemini API key: ")
#)

In [2]:
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

In [3]:
MODEL = "gemini-flash-latest"
TEMPERATURE_LIMIT = 80  # Illustrative threshold in degrees Celsius


def reactive_agent(temperature):
    """React to the current input, without retaining past readings."""

    # Rule-based assessment
    status = (
        "Temperature threshold exceeded"
        if temperature >= TEMPERATURE_LIMIT
        else "Temperature below the alert threshold"
    )

    # Gemini explains the assessment and recommends checks.
    prompt = f"""
You are a factory maintenance assistant.

Current input:
- Machine: Conveyor motor M-101
- Temperature: {temperature} degrees Celsius
- Demonstration threshold: {TEMPERATURE_LIMIT} degrees Celsius
- Assessment: {status}

Respond in three short points:
1. Explain the temperature assessment.
2. Recommend the next checks.
3. State what additional information would help.

Only the current temperature is available.
Do not assume a trend, confirm a fault, or claim actions were performed.
The threshold is illustrative, not a manufacturer specification.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    print(f"\nAssessment: {status}")
    print("Actual model used:", response.model_version)
    print("\nAgent response:")
    print(response.text or "No text response returned.")


print("Reactive maintenance agent")
print("Enter a temperature, or type 'exit' to stop.")

try:
    while True:
        user_input = input("\nMachine temperature in °C: ").strip()

        if user_input.lower() == "exit":
            print("Agent stopped.")
            break

        try:
            temperature = float(user_input)
        except ValueError:
            print("Enter a valid number, such as 65 or 88.")
            continue

        try:
            reactive_agent(temperature)
        except Exception as error:
            print(f"Gemini request failed: {type(error).__name__}: {error}")

finally:
    client.close()

Reactive maintenance agent
Enter a temperature, or type 'exit' to stop.

Machine temperature in °C: 67

Assessment: Temperature below the alert threshold
Actual model used: gemini-3.8-flash

Agent response:
1. **Temperature Assessment:** Conveyor motor M-101 is currently operating at 67.0 °C, which is 13.0 °C below the illustrative 80.0 °C demonstration threshold, placing it within the bounds of this reference limit.
2. **Recommended Next Checks:** Perform routine non-invasive checks, including inspecting the motor's cooling fins and fan cover for airflow obstructions, and checking for unusual vibration or audible noise.
3. **Additional Helpful Information:** Manufacturer-specified thermal ratings, ambient room temperature, electrical load data, and historical baseline temperatures would help determine if 67.0 °C is normal for this machine.

Machine temperature in °C: exit
Agent stopped.


In [4]:
# Re-initialize the client since it was closed in the previous cell's finally block
client = genai.Client(api_key=api_key)

test_prompt = "Hello! What model version are you?"

response = client.models.generate_content(
    model=MODEL,
    contents=test_prompt,
)

print("Requested model:", MODEL)
print("Actual model used:", response.model_version)
print("\nResponse text:\n", response.text)

Requested model: gemini-flash-latest
Actual model used: gemini-3.8-flash

Response text:
 Hello! I am Gemini 3.8 Flash, a large language model built by Google. How can I help you today?
